In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repository root to system path to import from src
sys.path.append("..")
from src.data import load_raw_data, clean_data, validate_schema

# Global seed for reproducibility
RANDOM_STATE = 42

# Ensure output directories exist
os.makedirs("../reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
# Load raw data and check shape and missing values
df_raw = load_raw_data("../data/hotel_bookings.csv")
print(f"Raw Data Shape: {df_raw.shape}")

print("\nMissing values count per column:")
missing_values = df_raw.isnull().sum()
print(missing_values[missing_values > 0])

In [ ]:
# Prove target leakage: Crosstab between is_canceled and reservation_status
leakage_crosstab = pd.crosstab(df_raw['is_canceled'], df_raw['reservation_status'], margins=True)
display(leakage_crosstab)

### Target Leakage Audit Note

As shown in the crosstab above, `reservation_status` has a **100% deterministic correlation** with the target variable `is_canceled`:
- Every booking with `is_canceled = 0` has `reservation_status = 'Check-Out'`.
- Every booking with `is_canceled = 1` has `reservation_status` in `['Canceled', 'No-Show']`.

`reservation_status` (and its counterpart `reservation_status_date`) is determined only **after** the booking lifecycle has concluded. In a real-world predictive setting, this information is not available at booking time. Including it would cause severe **target leakage**, resulting in an artificially inflated (near-perfect) model that fails in production. Therefore, `reservation_status` and `reservation_status_date` must be dropped.

In [ ]:
# Clean the data and validate schema
df_cleaned = clean_data(df_raw)
validate_schema(df_cleaned)

# Ensure processed directory exists and export cleaned dataset
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/hotel_bookings_cleaned.csv"
df_cleaned.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully exported to: {output_path}")

In [ ]:
# Cell 6: Plot Cancellations vs. Hotel Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df_cleaned, x='hotel', hue='is_canceled', palette='Set2')
plt.title("Cancellations by Hotel Type", fontsize=14)
plt.xlabel("Hotel Type", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Is Canceled", labels=["Not Canceled (0)", "Canceled (1)"])
plt.tight_layout()
plt.savefig("../reports/figures/cancellations_by_hotel.png", dpi=300)
plt.show()

In [ ]:
# Cell 7: Plot Cancellations vs. Lead Time
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_cleaned, x='is_canceled', y='lead_time', palette='Set2')
plt.title("Lead Time Distribution by Cancellation Status", fontsize=14)
plt.xlabel("Is Canceled (0 = No, 1 = Yes)", fontsize=12)
plt.ylabel("Lead Time (days)", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/figures/lead_time_vs_cancellation.png", dpi=300)
plt.show()

In [ ]:
# Cell 8: Plot Cancellations vs. Deposit Type
plt.figure(figsize=(8, 5))
sns.countplot(data=df_cleaned, x='deposit_type', hue='is_canceled', palette='Set2')
plt.title("Cancellations by Deposit Type", fontsize=14)
plt.xlabel("Deposit Type", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Is Canceled", labels=["Not Canceled (0)", "Canceled (1)"])
plt.tight_layout()
plt.savefig("../reports/figures/cancellations_by_deposit_type.png", dpi=300)
plt.show()

In [ ]:
# Cell 9: Plot Cancellations vs. Market Segment
plt.figure(figsize=(10, 6))
sns.countplot(data=df_cleaned, x='market_segment', hue='is_canceled', palette='Set2')
plt.title("Cancellations by Market Segment", fontsize=14)
plt.xlabel("Market Segment", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Is Canceled", labels=["Not Canceled (0)", "Canceled (1)"])
plt.tight_layout()
plt.savefig("../reports/figures/cancellations_by_market_segment.png", dpi=300)
plt.show()